# Pricing Project

## Question 1

In [7]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp

df = pd.read_csv('Data/data.csv')

features = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score', 'prop_accesibility_score',
    'prop_log_historical_price', 'price_usd', 'promotion_flag'
]

# Normalize features
for f in features:
    df[f] = (df[f] - df[f].mean()) / df[f].std()

# Group by search session
groups = []
for _, g in df.groupby('srch_id'):
    groups.append((g[features].values, g['booking_bool'].values))

def neg_ll(beta):
    b0, b = beta[0], beta[1:]
    nll, grad = 0.0, np.zeros_like(beta)
    for X, y in groups:
        u = b0 + X @ b
        ld = logsumexp(np.append(0.0, u))
        p = np.exp(u - ld)
        d = p - y
        nll += ld - y @ u
        grad[0] += d.sum()
        grad[1:] += X.T @ d
    return nll, grad

res = minimize(neg_ll, x0=np.zeros(9), jac=True, method='L-BFGS-B')
beta = res.x

print('Estimated MNL coefficients:')
for name, val in zip(['intercept'] + features, beta):
    print(f'  {name:35s} {val:+f}')

Estimated MNL coefficients:
  intercept                           -1.746680
  prop_starrating                     +0.408132
  prop_review_score                   +0.108754
  prop_brand_bool                     +0.101381
  prop_location_score                 +0.021995
  prop_accesibility_score             +0.043445
  prop_log_historical_price           -0.066872
  price_usd                           -1.331110
  promotion_flag                      +0.159765
